In [1]:
# ============================================================
# Hybrid GNN Implementation & Testing Notebook
# ============================================================
# This notebook tests the hybrid architecture modules on Colab
# Run cells sequentially to verify each module works correctly

print("🚀 Starting Hybrid GNN Development Environment")
print("=" * 50)

🚀 Starting Hybrid GNN Development Environment


# 📦 Step 1: Install Dependencies
Install required packages for Colab environment

In [2]:
# Install dependencies (run this cell first on Colab)
!pip install torch torch-geometric faiss-cpu numpy -q
!pip install pytest -q

# Verify installation
import torch
import torch_geometric
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ PyG version: {torch_geometric.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.3 MB/s eta 0:00:0000:0100:01m
✅ PyTorch version: 2.9.0+cpu
✅ PyG version: 2.7.0
✅ CUDA available: False


# 🧪 Step 2: Test HybridSampler
Testing the edge augmentation module with FAISS semantic search

In [3]:
# ============================================================
# HybridSampler Implementation (inline for Colab testing)
# ============================================================
"""
HybridSampler - Edge Augmentation for Knowledge Graphs

Creates augmented graphs with three types of edges:
1. Real edges: Original edges from the knowledge graph
2. Semantic edges: Edges between semantically similar nodes (via FAISS)
3. Random edges: Random node pairs for exploration

Edge Type Encoding:
    0 = Real edge (from original KG)
    1 = Semantic edge (from FAISS similarity)
    2 = Random edge (uniform sampling)
"""

import torch
import numpy as np
from typing import Tuple, Optional
from torch_geometric.data import Data

# Try to import FAISS
try:
    import faiss
    FAISS_AVAILABLE = True
    print("✅ FAISS is available")
except ImportError:
    FAISS_AVAILABLE = False
    print("⚠️ FAISS not available, using torch fallback")


class HybridSampler:
    """
    Sampler for creating augmented knowledge graphs with semantic and random edges.
    
    Attributes:
        k_semantic (int): Number of semantic neighbors per node
        k_random (int): Number of random edges per node
        use_faiss (bool): Whether to use FAISS or torch fallback
    """
    
    # Edge type constants
    EDGE_TYPE_REAL = 0
    EDGE_TYPE_SEMANTIC = 1
    EDGE_TYPE_RANDOM = 2
    
    def __init__(
        self,
        k_semantic: int = 10,
        k_random: int = 5,
        use_faiss: bool = True,
        faiss_index_type: str = "HNSW",
        faiss_hnsw_m: int = 32,
        seed: Optional[int] = None,
    ):
        if k_semantic < 0 or k_random < 0:
            raise ValueError("k_semantic and k_random must be non-negative")
        
        self.k_semantic = k_semantic
        self.k_random = k_random
        self.use_faiss = use_faiss and FAISS_AVAILABLE
        self.faiss_index_type = faiss_index_type
        self.faiss_hnsw_m = faiss_hnsw_m
        self.seed = seed
        
        self.index = None
        self.embeddings_cache = None
        self._num_nodes = None
        self._embedding_dim = None
        
        if seed is not None:
            np.random.seed(seed)
            torch.manual_seed(seed)
    
    def build_semantic_index(self, text_embeddings: torch.Tensor, normalize: bool = True) -> None:
        """Build FAISS index from text embeddings for semantic neighbor search."""
        self._num_nodes = text_embeddings.size(0)
        self._embedding_dim = text_embeddings.size(1)
        
        embeddings_np = text_embeddings.detach().cpu().numpy().astype('float32')
        
        if normalize:
            norms = np.linalg.norm(embeddings_np, axis=1, keepdims=True)
            norms = np.maximum(norms, 1e-8)
            embeddings_np = embeddings_np / norms
        
        if self.use_faiss:
            dim = embeddings_np.shape[1]
            
            if self.faiss_index_type == "HNSW":
                self.index = faiss.IndexHNSWFlat(dim, self.faiss_hnsw_m)
                self.index.hnsw.efConstruction = 200
                self.index.hnsw.efSearch = 50
            elif self.faiss_index_type == "Flat":
                self.index = faiss.IndexFlatIP(dim)
            else:
                self.index = faiss.IndexFlatIP(dim)
            
            self.index.add(embeddings_np)
        else:
            self.embeddings_cache = torch.from_numpy(embeddings_np)
    
    def _find_semantic_neighbors_faiss(self, text_embeddings: torch.Tensor) -> Tuple[np.ndarray, np.ndarray]:
        """Find k nearest neighbors using FAISS."""
        query_np = text_embeddings.detach().cpu().numpy().astype('float32')
        norms = np.linalg.norm(query_np, axis=1, keepdims=True)
        norms = np.maximum(norms, 1e-8)
        query_np = query_np / norms
        
        k_search = min(self.k_semantic + 1, self._num_nodes)
        distances, indices = self.index.search(query_np, k_search)
        
        num_nodes = text_embeddings.size(0)
        src_list, dst_list = [], []
        
        for i in range(num_nodes):
            neighbors = indices[i]
            valid_neighbors = [n for n in neighbors if n != i and n >= 0 and n < num_nodes]
            valid_neighbors = valid_neighbors[:self.k_semantic]
            for neighbor in valid_neighbors:
                src_list.append(i)
                dst_list.append(neighbor)
        
        return np.array(src_list), np.array(dst_list)
    
    def _find_semantic_neighbors_torch(self, text_embeddings: torch.Tensor) -> Tuple[np.ndarray, np.ndarray]:
        """Fallback: Find k nearest neighbors using torch cosine similarity."""
        embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=-1)
        similarity = torch.mm(embeddings_norm, embeddings_norm.t())
        similarity.fill_diagonal_(-float('inf'))
        
        k = min(self.k_semantic, similarity.size(1) - 1)
        _, top_k_indices = similarity.topk(k, dim=1)
        
        num_nodes = text_embeddings.size(0)
        src_indices = np.repeat(np.arange(num_nodes), k)
        dst_indices = top_k_indices.cpu().numpy().flatten()
        
        return src_indices, dst_indices
    
    def _sample_random_edges(self, num_nodes: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample random edges uniformly."""
        if self.k_random <= 0:
            return (torch.empty(0, dtype=torch.long, device=device),
                    torch.empty(0, dtype=torch.long, device=device))
        
        num_random = num_nodes * self.k_random
        src_random = torch.randint(0, num_nodes, (num_random,), device=device)
        dst_random = torch.randint(0, num_nodes, (num_random,), device=device)
        
        mask = src_random != dst_random
        return src_random[mask], dst_random[mask]
    
    def sample_edges(self, graph: Data, text_embeddings: torch.Tensor, 
                     add_reverse: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
        """Generate augmented edge_index and edge_type tensors."""
        if self.index is None and self.embeddings_cache is None:
            raise RuntimeError("Must call build_semantic_index() before sample_edges()")
        
        device = graph.edge_index.device
        num_nodes = graph.num_nodes
        
        # 1. Real edges
        real_edges = graph.edge_index
        num_real = real_edges.size(1)
        real_types = torch.full((num_real,), self.EDGE_TYPE_REAL, dtype=torch.long, device=device)
        
        # 2. Semantic edges
        if self.k_semantic > 0:
            if self.use_faiss and self.index is not None:
                src_sem, dst_sem = self._find_semantic_neighbors_faiss(text_embeddings)
            else:
                src_sem, dst_sem = self._find_semantic_neighbors_torch(text_embeddings)
            
            semantic_edges = torch.stack([
                torch.from_numpy(src_sem).to(device),
                torch.from_numpy(dst_sem).to(device),
            ])
            num_semantic = semantic_edges.size(1)
            semantic_types = torch.full((num_semantic,), self.EDGE_TYPE_SEMANTIC, dtype=torch.long, device=device)
        else:
            semantic_edges = torch.empty(2, 0, dtype=torch.long, device=device)
            semantic_types = torch.empty(0, dtype=torch.long, device=device)
        
        # 3. Random edges
        src_random, dst_random = self._sample_random_edges(num_nodes, device)
        if src_random.size(0) > 0:
            random_edges = torch.stack([src_random, dst_random])
            num_random = random_edges.size(1)
            random_types = torch.full((num_random,), self.EDGE_TYPE_RANDOM, dtype=torch.long, device=device)
        else:
            random_edges = torch.empty(2, 0, dtype=torch.long, device=device)
            random_types = torch.empty(0, dtype=torch.long, device=device)
        
        # 4. Combine
        aug_edge_index = torch.cat([real_edges, semantic_edges, random_edges], dim=1)
        aug_edge_type = torch.cat([real_types, semantic_types, random_types])
        
        if add_reverse:
            reverse_edge_index = aug_edge_index.flip(0)
            aug_edge_index = torch.cat([aug_edge_index, reverse_edge_index], dim=1)
            aug_edge_type = torch.cat([aug_edge_type, aug_edge_type])
        
        return aug_edge_index, aug_edge_type
    
    def get_edge_statistics(self, aug_edge_type: torch.Tensor) -> dict:
        """Get statistics about the augmented edges."""
        total = aug_edge_type.size(0)
        num_real = (aug_edge_type == self.EDGE_TYPE_REAL).sum().item()
        num_semantic = (aug_edge_type == self.EDGE_TYPE_SEMANTIC).sum().item()
        num_random = (aug_edge_type == self.EDGE_TYPE_RANDOM).sum().item()
        
        return {
            "total_edges": total,
            "real_edges": num_real,
            "semantic_edges": num_semantic,
            "random_edges": num_random,
            "real_pct": num_real / total * 100 if total > 0 else 0,
            "semantic_pct": num_semantic / total * 100 if total > 0 else 0,
            "random_pct": num_random / total * 100 if total > 0 else 0,
        }

print("✅ HybridSampler class defined")

✅ FAISS is available
✅ HybridSampler class defined


In [4]:
# ============================================================
# Test HybridSampler
# ============================================================

def test_hybrid_sampler():
    """Comprehensive test for HybridSampler"""
    print("🧪 Testing HybridSampler...")
    print("-" * 50)
    
    # Create test graph (10 nodes, grid-like structure)
    edge_index = torch.tensor([
        [0, 1, 1, 2, 0, 3, 1, 4, 2, 5, 3, 4, 4, 5, 3, 6, 4, 7, 5, 8, 7, 9],
        [1, 0, 2, 1, 3, 0, 4, 1, 5, 2, 4, 3, 5, 4, 6, 3, 7, 4, 8, 5, 9, 7],
    ], dtype=torch.long)
    graph = Data(edge_index=edge_index, num_nodes=10)
    
    # Create clustered embeddings (3 clusters + 1 isolated node)
    torch.manual_seed(42)
    dim = 32
    embeddings = torch.zeros(10, dim)
    
    # Cluster 1: nodes 0, 1, 2
    cluster1_base = torch.randn(dim)
    embeddings[0] = cluster1_base + torch.randn(dim) * 0.1
    embeddings[1] = cluster1_base + torch.randn(dim) * 0.1
    embeddings[2] = cluster1_base + torch.randn(dim) * 0.1
    
    # Cluster 2: nodes 3, 4, 5
    cluster2_base = torch.randn(dim)
    embeddings[3] = cluster2_base + torch.randn(dim) * 0.1
    embeddings[4] = cluster2_base + torch.randn(dim) * 0.1
    embeddings[5] = cluster2_base + torch.randn(dim) * 0.1
    
    # Cluster 3: nodes 6, 7, 8
    cluster3_base = torch.randn(dim)
    embeddings[6] = cluster3_base + torch.randn(dim) * 0.1
    embeddings[7] = cluster3_base + torch.randn(dim) * 0.1
    embeddings[8] = cluster3_base + torch.randn(dim) * 0.1
    
    # Node 9: isolated
    embeddings[9] = torch.randn(dim)
    
    # Test 1: Basic initialization
    print("Test 1: Initialization...")
    sampler = HybridSampler(k_semantic=3, k_random=2, seed=42)
    assert sampler.k_semantic == 3
    assert sampler.k_random == 2
    print("  ✅ Initialization passed")
    
    # Test 2: Build semantic index
    print("Test 2: Build semantic index...")
    sampler.build_semantic_index(embeddings)
    assert sampler._num_nodes == 10
    assert sampler._embedding_dim == 32
    print("  ✅ Semantic index built")
    
    # Test 3: Sample edges
    print("Test 3: Sample edges...")
    aug_edge_index, aug_edge_type = sampler.sample_edges(graph, embeddings)
    
    # Check shapes
    assert aug_edge_index.dim() == 2
    assert aug_edge_index.size(0) == 2
    assert aug_edge_type.dim() == 1
    assert aug_edge_index.size(1) == aug_edge_type.size(0)
    print(f"  Total edges: {aug_edge_index.size(1)}")
    print("  ✅ Edge sampling passed")
    
    # Test 4: Edge types are valid
    print("Test 4: Edge type validity...")
    assert (aug_edge_type >= 0).all()
    assert (aug_edge_type <= 2).all()
    print("  ✅ Edge types are valid (0, 1, or 2)")
    
    # Test 5: Real edges preserved
    print("Test 5: Real edges preservation...")
    num_real = (aug_edge_type == 0).sum().item()
    assert num_real == graph.edge_index.size(1), f"Expected {graph.edge_index.size(1)} real edges, got {num_real}"
    print(f"  Original edges: {graph.edge_index.size(1)}, Preserved: {num_real}")
    print("  ✅ Real edges preserved")
    
    # Test 6: No self-loops
    print("Test 6: No self-loops in augmented edges...")
    semantic_mask = aug_edge_type == 1
    semantic_edges = aug_edge_index[:, semantic_mask]
    semantic_self_loops = (semantic_edges[0] == semantic_edges[1]).sum().item()
    
    random_mask = aug_edge_type == 2
    random_edges = aug_edge_index[:, random_mask]
    random_self_loops = (random_edges[0] == random_edges[1]).sum().item()
    
    assert semantic_self_loops == 0, f"Found {semantic_self_loops} semantic self-loops"
    assert random_self_loops == 0, f"Found {random_self_loops} random self-loops"
    print("  ✅ No self-loops found")
    
    # Test 7: Valid node indices
    print("Test 7: Valid node indices...")
    assert aug_edge_index.min() >= 0
    assert aug_edge_index.max() < graph.num_nodes
    print("  ✅ All node indices are valid")
    
    # Test 8: Statistics
    print("Test 8: Edge statistics...")
    stats = sampler.get_edge_statistics(aug_edge_type)
    print(f"  Total: {stats['total_edges']}")
    print(f"  Real: {stats['real_edges']} ({stats['real_pct']:.1f}%)")
    print(f"  Semantic: {stats['semantic_edges']} ({stats['semantic_pct']:.1f}%)")
    print(f"  Random: {stats['random_edges']} ({stats['random_pct']:.1f}%)")
    print("  ✅ Statistics computed")
    
    # Test 9: Semantic clustering check
    print("Test 9: Semantic edges connect similar nodes...")
    cluster1_nodes = {0, 1, 2}
    cluster1_edges = 0
    for i in range(semantic_edges.size(1)):
        src, dst = semantic_edges[0, i].item(), semantic_edges[1, i].item()
        if src in cluster1_nodes and dst in cluster1_nodes:
            cluster1_edges += 1
    print(f"  Intra-cluster edges (nodes 0,1,2): {cluster1_edges}")
    assert cluster1_edges > 0, "No semantic edges within cluster 1"
    print("  ✅ Semantic clustering working")
    
    print("-" * 50)
    print("🎉 All HybridSampler tests passed!")
    return True

# Run tests
test_hybrid_sampler()

🧪 Testing HybridSampler...
--------------------------------------------------
Test 1: Initialization...
  ✅ Initialization passed
Test 2: Build semantic index...
  ✅ Semantic index built
Test 3: Sample edges...
  Total edges: 72
  ✅ Edge sampling passed
Test 4: Edge type validity...
  ✅ Edge types are valid (0, 1, or 2)
Test 5: Real edges preservation...
  Original edges: 22, Preserved: 22
  ✅ Real edges preserved
Test 6: No self-loops in augmented edges...
  ✅ No self-loops found
Test 7: Valid node indices...
  ✅ All node indices are valid
Test 8: Edge statistics...
  Total: 72
  Real: 22 (30.6%)
  Semantic: 30 (41.7%)
  Random: 20 (27.8%)
  ✅ Statistics computed
Test 9: Semantic edges connect similar nodes...
  Intra-cluster edges (nodes 0,1,2): 6
  ✅ Semantic clustering working
--------------------------------------------------
🎉 All HybridSampler tests passed!


True

# 🔮 Step 3: PEARL_GIN - Positional Encoding
Positional Encoding via Random Laplacian using Graph Isomorphism Network (GIN)

In [5]:
# ============================================================
# PEARL_GIN - Positional Encoding via Random Laplacian
# ============================================================
"""
PEARL (Positional Encoding via Random Laplacian)

Why PEARL?
----------
- Traditional positional encodings (e.g., Laplacian eigenvectors) are expensive to compute
- PEARL uses random noise + GNN propagation to learn structural information
- Isolated nodes get unique PE (from random noise) instead of being ignored
- Permutation equivariant (desirable property for GNNs)

Architecture:
    Input: Random Gaussian Noise [num_nodes, input_dim]
           + Real Edges only (no semantic/random)
    
    Process: 2-layer GINConv with BatchNorm and ReLU
    
    Output: Structural Positional Encoding [num_nodes, hidden_dim]

Key Properties:
    - Runs ONLY on real edges (structural information only)
    - Noise is unique per node → unique PE even for isolated nodes
    - Lightweight: 2 layers of GIN (fast to compute)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv


class PEARL_GIN(nn.Module):
    """
    Positional Encoding via Random Laplacian using GIN.
    
    This module learns structural positional encodings by propagating
    random noise through the graph structure using GIN layers.
    
    Attributes:
        input_dim (int): Dimension of input random noise
        hidden_dim (int): Dimension of output positional encoding
        num_layers (int): Number of GIN layers (default: 2)
        dropout (float): Dropout probability (default: 0.1)
    
    Example:
        >>> pearl = PEARL_GIN(input_dim=32, hidden_dim=32)
        >>> noise = torch.randn(100, 32)  # 100 nodes
        >>> h_pos = pearl(noise, edge_index)  # [100, 32]
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int = 2,
        dropout: float = 0.1,
        eps: float = 0.0,
        train_eps: bool = True,
    ):
        """
        Initialize PEARL_GIN.
        
        Args:
            input_dim: Dimension of input random noise
            hidden_dim: Dimension of hidden layers and output
            num_layers: Number of GIN layers (recommended: 2-3)
            dropout: Dropout probability between layers
            eps: Initial epsilon value for GIN (learnable if train_eps=True)
            train_eps: Whether to make epsilon trainable
        """
        super(PEARL_GIN, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        
        # GIN layers with MLP
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            
            # MLP for GIN: 2-layer MLP with hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            
            # GINConv with learnable epsilon
            self.convs.append(GINConv(mlp, eps=eps, train_eps=train_eps))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        
        # Final projection (optional, for dimension matching)
        self.final_proj = nn.Linear(hidden_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights for stable training."""
        for conv in self.convs:
            for module in conv.nn.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
    
    def forward(
        self,
        x_noise: torch.Tensor,
        edge_index: torch.Tensor,
        return_all_layers: bool = False,
    ) -> torch.Tensor:
        """
        Compute positional encodings from random noise.
        
        Args:
            x_noise: Random noise tensor [num_nodes, input_dim]
                    Should be sampled from standard Gaussian N(0, 1)
            edge_index: Graph connectivity [2, num_edges]
                       Should be REAL edges only (not augmented)
            return_all_layers: If True, return list of all layer outputs
        
        Returns:
            h_pos: Positional encoding [num_nodes, hidden_dim]
                  Or list of encodings if return_all_layers=True
        
        Note:
            - For isolated nodes (no edges), the output will be a transformation
              of the original noise, providing a unique identifier.
            - The random noise should be generated fresh for each forward pass
              during training, but can be fixed during inference for consistency.
        """
        h = x_noise
        layer_outputs = []
        
        for i, (conv, bn) in enumerate(zip(self.convs, self.batch_norms)):
            # GIN convolution
            h = conv(h, edge_index)
            
            # Batch normalization
            h = bn(h)
            
            # Activation
            h = F.relu(h)
            
            # Dropout (only during training, not on last layer)
            if i < self.num_layers - 1:
                h = F.dropout(h, p=self.dropout, training=self.training)
            
            layer_outputs.append(h)
        
        # Final projection
        h = self.final_proj(h)
        
        if return_all_layers:
            return layer_outputs
        
        return h
    
    def generate_noise(
        self,
        num_nodes: int,
        device: torch.device = None,
        seed: int = None,
    ) -> torch.Tensor:
        """
        Generate random noise for positional encoding.
        
        Args:
            num_nodes: Number of nodes in the graph
            device: Target device for the tensor
            seed: Random seed for reproducibility (optional)
        
        Returns:
            noise: Random Gaussian noise [num_nodes, input_dim]
        """
        if seed is not None:
            torch.manual_seed(seed)
        
        noise = torch.randn(num_nodes, self.input_dim, device=device)
        return noise
    
    def __repr__(self) -> str:
        return (
            f"PEARL_GIN("
            f"input_dim={self.input_dim}, "
            f"hidden_dim={self.hidden_dim}, "
            f"num_layers={self.num_layers}, "
            f"dropout={self.dropout})"
        )


print("✅ PEARL_GIN class defined")

✅ PEARL_GIN class defined


In [6]:
# ============================================================
# Test PEARL_GIN
# ============================================================

def test_pearl_gin():
    """Comprehensive test for PEARL_GIN module"""
    print("🧪 Testing PEARL_GIN...")
    print("-" * 50)
    
    # Create test graph
    edge_index = torch.tensor([
        [0, 1, 1, 2, 2, 3, 4, 5, 5, 6],  # Node 7, 8, 9 are isolated
        [1, 0, 2, 1, 3, 2, 5, 4, 6, 5],
    ], dtype=torch.long)
    num_nodes = 10
    
    # Test 1: Initialization
    print("Test 1: Initialization...")
    pearl = PEARL_GIN(input_dim=32, hidden_dim=32, num_layers=2)
    assert pearl.input_dim == 32
    assert pearl.hidden_dim == 32
    assert pearl.num_layers == 2
    print(f"  Model: {pearl}")
    print("  ✅ Initialization passed")
    
    # Test 2: Forward pass
    print("Test 2: Forward pass...")
    noise = torch.randn(num_nodes, 32)
    h_pos = pearl(noise, edge_index)
    
    assert h_pos.shape == (num_nodes, 32), f"Expected shape (10, 32), got {h_pos.shape}"
    print(f"  Output shape: {h_pos.shape}")
    print("  ✅ Forward pass passed")
    
    # Test 3: Isolated nodes get non-zero PE
    print("Test 3: Isolated nodes handling...")
    isolated_nodes = [7, 8, 9]  # No edges
    connected_nodes = [0, 1, 2, 3, 4, 5, 6]  # Have edges
    
    isolated_pe = h_pos[isolated_nodes]
    connected_pe = h_pos[connected_nodes]
    
    # Isolated nodes should have non-zero PE (from noise transformation)
    assert not torch.allclose(isolated_pe, torch.zeros_like(isolated_pe), atol=1e-6), \
        "Isolated nodes have zero PE!"
    print(f"  Isolated nodes PE norm: {isolated_pe.norm(dim=1).mean():.4f}")
    print(f"  Connected nodes PE norm: {connected_pe.norm(dim=1).mean():.4f}")
    print("  ✅ Isolated nodes have non-zero PE")
    
    # Test 4: PE uniqueness
    print("Test 4: PE uniqueness...")
    torch.manual_seed(42)
    noise1 = torch.randn(num_nodes, 32)
    h_pos1 = pearl(noise1, edge_index)
    
    torch.manual_seed(123)  # Different seed
    noise2 = torch.randn(num_nodes, 32)
    h_pos2 = pearl(noise2, edge_index)
    
    # Different noise should give different PE
    assert not torch.allclose(h_pos1, h_pos2, atol=1e-4), \
        "Different noise gives same PE!"
    print("  ✅ Different noise produces different PE")
    
    # Test 5: Gradient flow
    print("Test 5: Gradient flow (backward pass)...")
    pearl.train()
    noise = torch.randn(num_nodes, 32, requires_grad=True)
    h_pos = pearl(noise, edge_index)
    
    loss = h_pos.sum()
    loss.backward()
    
    assert noise.grad is not None, "No gradients computed!"
    assert not torch.isnan(noise.grad).any(), "NaN in gradients!"
    print(f"  Gradient norm: {noise.grad.norm():.4f}")
    print("  ✅ Gradient flow working")
    
    # Test 6: Return all layers
    print("Test 6: Return all layers...")
    noise = torch.randn(num_nodes, 32)
    layer_outputs = pearl(noise, edge_index, return_all_layers=True)
    
    assert len(layer_outputs) == 2, f"Expected 2 layers, got {len(layer_outputs)}"
    for i, out in enumerate(layer_outputs):
        print(f"  Layer {i+1} output shape: {out.shape}")
    print("  ✅ All layer outputs returned")
    
    # Test 7: Generate noise helper
    print("Test 7: Noise generation helper...")
    noise = pearl.generate_noise(num_nodes=50, seed=42)
    assert noise.shape == (50, 32)
    
    noise_same = pearl.generate_noise(num_nodes=50, seed=42)
    assert torch.allclose(noise, noise_same), "Same seed gives different noise!"
    print("  ✅ Noise generation working")
    
    # Test 8: Different number of layers
    print("Test 8: Different configurations...")
    pearl_1layer = PEARL_GIN(input_dim=64, hidden_dim=32, num_layers=1)
    pearl_3layer = PEARL_GIN(input_dim=16, hidden_dim=64, num_layers=3)
    
    noise_64 = torch.randn(num_nodes, 64)
    noise_16 = torch.randn(num_nodes, 16)
    
    out_1 = pearl_1layer(noise_64, edge_index)
    out_3 = pearl_3layer(noise_16, edge_index)
    
    assert out_1.shape == (num_nodes, 32)
    assert out_3.shape == (num_nodes, 64)
    print(f"  1-layer (64→32): {out_1.shape}")
    print(f"  3-layer (16→64): {out_3.shape}")
    print("  ✅ Different configurations work")
    
    # Test 9: GPU compatibility (if available)
    if torch.cuda.is_available():
        print("Test 9: GPU compatibility...")
        pearl_gpu = pearl.cuda()
        noise_gpu = torch.randn(num_nodes, 32, device='cuda')
        edge_index_gpu = edge_index.cuda()
        
        h_pos_gpu = pearl_gpu(noise_gpu, edge_index_gpu)
        assert h_pos_gpu.device.type == 'cuda'
        print("  ✅ GPU forward pass working")
    else:
        print("Test 9: GPU test skipped (CUDA not available)")
    
    print("-" * 50)
    print("🎉 All PEARL_GIN tests passed!")
    return True

# Run tests
test_pearl_gin()

🧪 Testing PEARL_GIN...
--------------------------------------------------
Test 1: Initialization...
  Model: PEARL_GIN(input_dim=32, hidden_dim=32, num_layers=2, dropout=0.1)
  ✅ Initialization passed
Test 2: Forward pass...
  Output shape: torch.Size([10, 32])
  ✅ Forward pass passed
Test 3: Isolated nodes handling...
  Isolated nodes PE norm: 3.3271
  Connected nodes PE norm: 4.1477
  ✅ Isolated nodes have non-zero PE
Test 4: PE uniqueness...
  ✅ Different noise produces different PE
Test 5: Gradient flow (backward pass)...
  Gradient norm: 20.9654
  ✅ Gradient flow working
Test 6: Return all layers...
  Layer 1 output shape: torch.Size([10, 32])
  Layer 2 output shape: torch.Size([10, 32])
  ✅ All layer outputs returned
Test 7: Noise generation helper...
  ✅ Noise generation working
Test 8: Different configurations...
  1-layer (64→32): torch.Size([10, 32])
  3-layer (16→64): torch.Size([10, 64])
  ✅ Different configurations work
Test 9: GPU test skipped (CUDA not available)
-------

True

# ⚡ Step 4: SparseGTConv - Sparse Graph Transformer
Multi-head attention on sparse augmented graph with edge-type awareness

In [7]:
# ============================================================
# SparseGTConv - Sparse Graph Transformer Layer
# ============================================================
"""
SparseGTConv - Graph Transformer with Sparse Attention

Why Sparse GT instead of Full Attention?
-----------------------------------------
- Full attention is O(N²) → infeasible for large graphs
- Sparse attention only attends to neighbors → O(E) complexity
- Edge-type awareness allows different attention patterns for different edge types

Architecture:
    Input: Node features [num_nodes, in_channels]
           + Augmented edges [2, num_edges] (real + semantic + random)
           + Edge types [num_edges] (0=real, 1=semantic, 2=random)
    
    Multi-Head Attention:
        Q = Linear(x)  # Query
        K = Linear(x)  # Key  
        V = Linear(x)  # Value
        
        Edge Bias = Embedding(edge_type)
        
        Attention = softmax((Q_i · K_j + Q_i · EdgeBias) / sqrt(d))
        Output = Σ (Attention * V_j)
    
    Output: Updated node features [num_nodes, out_channels]

Edge Type Semantics in Attention:
---------------------------------
- Real edges (type 0): High weight → structural neighbors
- Semantic edges (type 1): Medium weight → similar content
- Random edges (type 2): Low weight → exploration

Note: FlashAttention can be integrated here for memory efficiency,
but we use standard attention for compatibility.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


class SparseGTConv(MessagePassing):
    """
    Sparse Graph Transformer Convolution Layer.
    
    Performs multi-head self-attention over the graph edges with
    edge-type aware attention biases.
    
    Attributes:
        in_channels (int): Input feature dimension
        out_channels (int): Output feature dimension
        num_heads (int): Number of attention heads
        num_edge_types (int): Number of edge types (default: 3)
        dropout (float): Dropout probability for attention weights
    
    Example:
        >>> layer = SparseGTConv(in_channels=64, out_channels=64, num_heads=4)
        >>> out = layer(x, edge_index, edge_type)
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_heads: int = 4,
        num_edge_types: int = 3,
        dropout: float = 0.1,
        bias: bool = True,
        add_self_loops: bool = False,
        **kwargs,
    ):
        """
        Initialize SparseGTConv.
        
        Args:
            in_channels: Dimension of input node features
            out_channels: Dimension of output node features
            num_heads: Number of attention heads (out_channels must be divisible by num_heads)
            num_edge_types: Number of distinct edge types (3 for real/semantic/random)
            dropout: Dropout probability for attention weights
            bias: Whether to use bias in linear projections
            add_self_loops: Whether to add self-loops (usually False for GT)
        """
        # aggregation method for MessagePassing
        kwargs.setdefault('aggr', 'add')
        super(SparseGTConv, self).__init__(node_dim=0, **kwargs)
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_heads = num_heads
        self.num_edge_types = num_edge_types
        self.dropout = dropout
        self.add_self_loops = add_self_loops
        
        # Check divisibility
        assert out_channels % num_heads == 0, \
            f"out_channels ({out_channels}) must be divisible by num_heads ({num_heads})"
        
        self.head_dim = out_channels // num_heads
        self.scale = self.head_dim ** -0.5  # 1/sqrt(d) for scaled attention
        
        # Linear projections for Q, K, V
        self.lin_q = nn.Linear(in_channels, out_channels, bias=bias)
        self.lin_k = nn.Linear(in_channels, out_channels, bias=bias)
        self.lin_v = nn.Linear(in_channels, out_channels, bias=bias)
        
        # Edge type embedding for attention bias
        # Each edge type has its own embedding that modifies attention scores
        self.edge_type_emb = nn.Embedding(num_edge_types, out_channels)
        
        # Output projection
        self.lin_out = nn.Linear(out_channels, out_channels, bias=bias)
        
        # Layer normalization (pre-norm style)
        self.norm1 = nn.LayerNorm(in_channels)
        self.norm2 = nn.LayerNorm(out_channels)
        
        # Feed-forward network (FFN) after attention
        self.ffn = nn.Sequential(
            nn.Linear(out_channels, out_channels * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_channels * 4, out_channels),
            nn.Dropout(dropout),
        )
        
        # Skip connection projection (if dimensions don't match)
        self.skip_proj = nn.Linear(in_channels, out_channels, bias=False) \
            if in_channels != out_channels else nn.Identity()
        
        self._reset_parameters()
    
    def _reset_parameters(self):
        """Initialize parameters for stable training."""
        nn.init.xavier_uniform_(self.lin_q.weight)
        nn.init.xavier_uniform_(self.lin_k.weight)
        nn.init.xavier_uniform_(self.lin_v.weight)
        nn.init.xavier_uniform_(self.lin_out.weight)
        
        # Initialize edge embeddings near zero (small bias initially)
        nn.init.normal_(self.edge_type_emb.weight, mean=0.0, std=0.02)
        
        if self.lin_q.bias is not None:
            nn.init.zeros_(self.lin_q.bias)
            nn.init.zeros_(self.lin_k.bias)
            nn.init.zeros_(self.lin_v.bias)
            nn.init.zeros_(self.lin_out.bias)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_type: torch.Tensor,
        return_attention: bool = False,
    ):
        """
        Forward pass of sparse graph transformer.
        
        Args:
            x: Node features [num_nodes, in_channels]
            edge_index: Graph connectivity [2, num_edges]
            edge_type: Edge type labels [num_edges] with values in {0, 1, 2}
            return_attention: If True, also return attention weights
        
        Returns:
            out: Updated node features [num_nodes, out_channels]
            attn_weights: (optional) Attention weights [num_edges, num_heads]
        """
        num_nodes = x.size(0)
        
        # Pre-normalization
        x_norm = self.norm1(x)
        
        # Compute Q, K, V projections
        # Shape: [num_nodes, num_heads, head_dim]
        q = self.lin_q(x_norm).view(-1, self.num_heads, self.head_dim)
        k = self.lin_k(x_norm).view(-1, self.num_heads, self.head_dim)
        v = self.lin_v(x_norm).view(-1, self.num_heads, self.head_dim)
        
        # Store for attention computation
        self._attn_weights = None
        
        # Message passing
        out = self.propagate(
            edge_index,
            q=q, k=k, v=v,
            edge_type=edge_type,
            size=None,
        )
        
        # Reshape: [num_nodes, num_heads, head_dim] -> [num_nodes, out_channels]
        out = out.view(-1, self.out_channels)
        
        # Output projection
        out = self.lin_out(out)
        out = F.dropout(out, p=self.dropout, training=self.training)
        
        # Residual connection (skip)
        out = out + self.skip_proj(x)
        
        # Post-attention normalization and FFN
        out_norm = self.norm2(out)
        out = out + self.ffn(out_norm)
        
        if return_attention:
            return out, self._attn_weights
        
        return out
    
    def message(
        self,
        q_i: torch.Tensor,
        k_j: torch.Tensor,
        v_j: torch.Tensor,
        edge_type: torch.Tensor,
        index: torch.Tensor,
        ptr,
        size_i,
    ) -> torch.Tensor:
        """
        Compute attention-weighted messages.
        
        This method is called by MessagePassing.propagate() for each edge.
        
        Args:
            q_i: Query vectors for target nodes [num_edges, num_heads, head_dim]
            k_j: Key vectors for source nodes [num_edges, num_heads, head_dim]
            v_j: Value vectors for source nodes [num_edges, num_heads, head_dim]
            edge_type: Edge type labels [num_edges]
            index: Target node indices for softmax grouping
            ptr: CSR format pointers (optional)
            size_i: Number of target nodes
        
        Returns:
            messages: Attention-weighted value vectors [num_edges, num_heads, head_dim]
        """
        # Get edge type embeddings
        edge_emb = self.edge_type_emb(edge_type)  # [num_edges, out_channels]
        edge_emb = edge_emb.view(-1, self.num_heads, self.head_dim)  # [num_edges, num_heads, head_dim]
        
        # Compute attention scores
        # Standard attention: (Q_i · K_j) / sqrt(d)
        attn = (q_i * k_j).sum(dim=-1) * self.scale  # [num_edges, num_heads]
        
        # Add edge type bias: (Q_i · EdgeEmb) / sqrt(d)
        edge_bias = (q_i * edge_emb).sum(dim=-1) * self.scale  # [num_edges, num_heads]
        attn = attn + edge_bias
        
        # Softmax over neighbors (per target node)
        attn = softmax(attn, index, ptr, size_i)  # [num_edges, num_heads]
        
        # Store attention weights for visualization
        self._attn_weights = attn.detach()
        
        # Apply dropout to attention weights
        attn = F.dropout(attn, p=self.dropout, training=self.training)
        
        # Weight values by attention
        # [num_edges, num_heads, 1] * [num_edges, num_heads, head_dim]
        return attn.unsqueeze(-1) * v_j
    
    def __repr__(self) -> str:
        return (
            f"SparseGTConv("
            f"in={self.in_channels}, "
            f"out={self.out_channels}, "
            f"heads={self.num_heads}, "
            f"edge_types={self.num_edge_types})"
        )


print("✅ SparseGTConv class defined")

✅ SparseGTConv class defined


In [8]:
# ============================================================
# Test SparseGTConv
# ============================================================

def test_sparse_gt_conv():
    """Comprehensive test for SparseGTConv module"""
    print("🧪 Testing SparseGTConv...")
    print("-" * 50)
    
    # Create test data
    num_nodes = 20
    in_channels = 64
    out_channels = 64
    num_edges = 100
    
    torch.manual_seed(42)
    
    # Random node features
    x = torch.randn(num_nodes, in_channels)
    
    # Random edges (with 3 types)
    src = torch.randint(0, num_nodes, (num_edges,))
    dst = torch.randint(0, num_nodes, (num_edges,))
    edge_index = torch.stack([src, dst])
    
    # Edge types: 0=real, 1=semantic, 2=random
    edge_type = torch.randint(0, 3, (num_edges,))
    
    # Test 1: Initialization
    print("Test 1: Initialization...")
    layer = SparseGTConv(
        in_channels=in_channels,
        out_channels=out_channels,
        num_heads=4,
        num_edge_types=3,
        dropout=0.1,
    )
    print(f"  Model: {layer}")
    
    # Count parameters
    num_params = sum(p.numel() for p in layer.parameters())
    print(f"  Parameters: {num_params:,}")
    print("  ✅ Initialization passed")
    
    # Test 2: Forward pass
    print("Test 2: Forward pass...")
    layer.eval()  # No dropout for deterministic test
    out = layer(x, edge_index, edge_type)
    
    assert out.shape == (num_nodes, out_channels), \
        f"Expected shape ({num_nodes}, {out_channels}), got {out.shape}"
    print(f"  Input shape: {x.shape}")
    print(f"  Output shape: {out.shape}")
    print("  ✅ Forward pass passed")
    
    # Test 3: No NaN in output
    print("Test 3: Output validity...")
    assert not torch.isnan(out).any(), "NaN in output!"
    assert not torch.isinf(out).any(), "Inf in output!"
    print(f"  Output range: [{out.min():.4f}, {out.max():.4f}]")
    print("  ✅ No NaN/Inf in output")
    
    # Test 4: Attention weights returned
    print("Test 4: Attention weights...")
    out, attn_weights = layer(x, edge_index, edge_type, return_attention=True)
    
    assert attn_weights is not None
    assert attn_weights.shape == (num_edges, 4)  # [num_edges, num_heads]
    print(f"  Attention shape: {attn_weights.shape}")
    print(f"  Attention range: [{attn_weights.min():.4f}, {attn_weights.max():.4f}]")
    print("  ✅ Attention weights returned")
    
    # Test 5: Attention weights sum to ~1 per target node
    print("Test 5: Attention normalization...")
    # Group by target node and sum
    target_nodes = edge_index[1]  # Destination nodes
    for node_idx in range(min(5, num_nodes)):  # Check first 5 nodes
        mask = target_nodes == node_idx
        if mask.sum() > 0:
            attn_sum = attn_weights[mask, 0].sum()  # First head
            # Should be close to 1.0 (softmax)
            if mask.sum() > 1:  # Only check if node has multiple incoming edges
                assert abs(attn_sum - 1.0) < 0.1, f"Attention sum for node {node_idx}: {attn_sum:.4f}"
    print("  ✅ Attention properly normalized")
    
    # Test 6: Edge type affects attention
    print("Test 6: Edge type influence...")
    # Compare attention for different edge types
    real_mask = edge_type == 0
    semantic_mask = edge_type == 1
    random_mask = edge_type == 2
    
    if real_mask.sum() > 0 and semantic_mask.sum() > 0:
        attn_real = attn_weights[real_mask].mean()
        attn_semantic = attn_weights[semantic_mask].mean()
        print(f"  Mean attention (real edges): {attn_real:.4f}")
        print(f"  Mean attention (semantic edges): {attn_semantic:.4f}")
    print("  ✅ Edge types processed")
    
    # Test 7: Gradient flow
    print("Test 7: Gradient flow...")
    layer.train()
    x_grad = x.clone().requires_grad_(True)
    out = layer(x_grad, edge_index, edge_type)
    
    loss = out.sum()
    loss.backward()
    
    assert x_grad.grad is not None
    assert not torch.isnan(x_grad.grad).any()
    print(f"  Input gradient norm: {x_grad.grad.norm():.4f}")
    print("  ✅ Gradient flow working")
    
    # Test 8: Different configurations
    print("Test 8: Different configurations...")
    configs = [
        {"in_channels": 32, "out_channels": 64, "num_heads": 8},
        {"in_channels": 128, "out_channels": 128, "num_heads": 4},
        {"in_channels": 64, "out_channels": 32, "num_heads": 2},
    ]
    
    for cfg in configs:
        test_layer = SparseGTConv(**cfg, num_edge_types=3)
        test_x = torch.randn(num_nodes, cfg["in_channels"])
        test_out = test_layer(test_x, edge_index, edge_type)
        assert test_out.shape == (num_nodes, cfg["out_channels"])
        print(f"  {cfg['in_channels']}→{cfg['out_channels']}, {cfg['num_heads']} heads: ✓")
    print("  ✅ Different configurations work")
    
    # Test 9: Empty edges handling
    print("Test 9: Edge cases...")
    # Graph with very few edges
    sparse_edge_index = torch.tensor([[0, 1], [1, 0]], dtype=torch.long)
    sparse_edge_type = torch.tensor([0, 0], dtype=torch.long)
    sparse_out = layer(x, sparse_edge_index, sparse_edge_type)
    assert sparse_out.shape == (num_nodes, out_channels)
    print("  Sparse graph (2 edges): ✓")
    print("  ✅ Edge cases handled")
    
    # Test 10: GPU compatibility (if available)
    if torch.cuda.is_available():
        print("Test 10: GPU compatibility...")
        layer_gpu = layer.cuda()
        x_gpu = x.cuda()
        edge_index_gpu = edge_index.cuda()
        edge_type_gpu = edge_type.cuda()
        
        out_gpu = layer_gpu(x_gpu, edge_index_gpu, edge_type_gpu)
        assert out_gpu.device.type == 'cuda'
        print("  ✅ GPU forward pass working")
    else:
        print("Test 10: GPU test skipped (CUDA not available)")
    
    print("-" * 50)
    print("🎉 All SparseGTConv tests passed!")
    return True

# Run tests
test_sparse_gt_conv()

🧪 Testing SparseGTConv...
--------------------------------------------------
Test 1: Initialization...
  Model: SparseGTConv(in=64, out=64, heads=4, edge_types=3)
  Parameters: 50,176
  ✅ Initialization passed
Test 2: Forward pass...
  Input shape: torch.Size([20, 64])
  Output shape: torch.Size([20, 64])
  ✅ Forward pass passed
Test 3: Output validity...
  Output range: [-4.2191, 3.6694]
  ✅ No NaN/Inf in output
Test 4: Attention weights...
  Attention shape: torch.Size([100, 4])
  Attention range: [0.0010, 0.9285]
  ✅ Attention weights returned
Test 5: Attention normalization...
  ✅ Attention properly normalized
Test 6: Edge type influence...
  Mean attention (real edges): 0.2004
  Mean attention (semantic edges): 0.2085
  ✅ Edge types processed
Test 7: Gradient flow...
  Input gradient norm: 58.9937
  ✅ Gradient flow working
Test 8: Different configurations...
  32→64, 8 heads: ✓
  128→128, 4 heads: ✓
  64→32, 2 heads: ✓
  ✅ Different configurations work
Test 9: Edge cases...
  Spar

True

# 🔗 Step 5: HybridBlock - Combining Local + Global
Parallel processing with Local PNA (structural) + Global GT (semantic)

In [12]:
# ============================================================
# HybridBlock - Combined Local + Global Processing
# ============================================================
"""
HybridBlock - The Core of Hybrid Architecture

Why Hybrid?
-----------
- Local PNA: Captures structural patterns from real KG edges
  → Good at: Multi-hop reasoning, relation patterns
  → Weak at: Long-range dependencies, isolated nodes
  
- Global GT: Captures semantic relationships from augmented edges
  → Good at: Semantic similarity, long-range connections
  → Weak at: Fine-grained structural details

Combined = Best of Both Worlds

Architecture:
    Input: x [num_nodes, hidden_dim]
           + real_edge_index (for PNA)
           + aug_edge_index (for GT)
           + query (relation embedding)
    
    ┌─────────────────────────────────┐
    │ LOCAL BRANCH: Lightweight PNA   │
    │   - Aggregators: mean, max      │
    │   - Runs on real_edge_index     │
    │   - Query-conditioned           │
    └───────────────┬─────────────────┘
                    │
    ┌───────────────┴─────────────────┐
    │ GLOBAL BRANCH: SparseGTConv     │
    │   - Multi-head attention        │
    │   - Runs on aug_edge_index      │
    │   - Edge-type aware             │
    └───────────────┬─────────────────┘
                    │
    ┌───────────────┴─────────────────┐
    │ FUSION: x + local + global      │
    │   - Residual connection         │
    │   - LayerNorm                   │
    └─────────────────────────────────┘
    
    Output: Updated features [num_nodes, hidden_dim]
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree, scatter


class LightweightPNA(MessagePassing):
    """
    Lightweight PNA (Principal Neighbourhood Aggregation) layer.
    
    Simplified version with only mean and max aggregators.
    Query-conditioned for relation-aware message passing.
    
    Attributes:
        hidden_dim (int): Hidden dimension
        num_heads (int): Number of attention heads for query conditioning
    """
    
    def __init__(
        self,
        hidden_dim: int,
        dropout: float = 0.1,
        **kwargs,
    ):
        kwargs.setdefault('aggr', None)  # Custom aggregation
        super(LightweightPNA, self).__init__(node_dim=0, **kwargs)
        
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        
        # Message MLP
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # Query projection for conditioning
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        
        # Aggregation combination (mean + max → hidden_dim)
        self.agg_combine = nn.Linear(hidden_dim * 2, hidden_dim)
        
        # Output projection with residual
        self.out_proj = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        self._reset_parameters()
    
    def _reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        query: torch.Tensor = None,
        batch: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Forward pass of lightweight PNA.
        
        Args:
            x: Node features [num_nodes, hidden_dim]
            edge_index: Graph connectivity [2, num_edges] (REAL edges only)
            query: Relation query for conditioning [batch_size, hidden_dim] or [num_nodes, hidden_dim]
            batch: Batch assignment for nodes (optional)
        
        Returns:
            out: Updated node features [num_nodes, hidden_dim]
        """
        num_nodes = x.size(0)
        
        # Expand query to all nodes if batch-level
        if query is not None:
            if query.size(0) != num_nodes:
                # query is [batch_size, hidden_dim], need to expand
                if batch is not None:
                    query = query[batch]  # [num_nodes, hidden_dim]
                else:
                    # Assume single batch, broadcast
                    query = query.expand(num_nodes, -1)
            query = self.query_proj(query)
        else:
            query = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        
        # Compute degree for normalization
        row, col = edge_index
        deg = degree(row, num_nodes, dtype=x.dtype).clamp(min=1)
        deg_inv_sqrt = deg.pow(-0.5)
        
        # Message passing
        out = self.propagate(
            edge_index,
            x=x,
            query=query,
            deg_inv_sqrt=deg_inv_sqrt,
        )
        
        # Output projection with residual
        out = self.out_proj(out)
        out = out + x  # Residual
        
        return out
    
    def message(
        self,
        x_i: torch.Tensor,
        x_j: torch.Tensor,
        query_i: torch.Tensor,
        deg_inv_sqrt_i: torch.Tensor,
        deg_inv_sqrt_j: torch.Tensor,
    ) -> torch.Tensor:
        """Compute messages with query conditioning."""
        # Concatenate source and query
        msg_input = torch.cat([x_j, query_i], dim=-1)
        
        # MLP on message
        msg = self.msg_mlp(msg_input)
        
        # Normalize by degree (symmetric normalization)
        norm = deg_inv_sqrt_i.unsqueeze(-1) * deg_inv_sqrt_j.unsqueeze(-1)
        msg = msg * norm
        
        return msg
    
    def aggregate(
        self,
        inputs: torch.Tensor,
        index: torch.Tensor,
        dim_size: int = None,
    ) -> torch.Tensor:
        """Aggregate messages using mean and max."""
        # Use PyG's scatter (no external torch_scatter needed)
        agg_mean = scatter(inputs, index, dim=0, dim_size=dim_size, reduce='mean')
        agg_max = scatter(inputs, index, dim=0, dim_size=dim_size, reduce='max')
        
        # Combine aggregations
        combined = torch.cat([agg_mean, agg_max], dim=-1)
        out = self.agg_combine(combined)
        
        return out


class HybridBlock(nn.Module):
    """
    Hybrid processing block combining Local PNA and Global Sparse GT.
    
    This is the core building block of the hybrid architecture.
    It processes the graph through two parallel branches and fuses
    the results with residual connections.
    
    Attributes:
        hidden_dim (int): Hidden dimension
        num_heads (int): Number of attention heads for GT
        num_edge_types (int): Number of edge types (default: 3)
        dropout (float): Dropout probability
        use_local (bool): Whether to use local PNA branch
        use_global (bool): Whether to use global GT branch
    
    Example:
        >>> block = HybridBlock(hidden_dim=64, num_heads=4)
        >>> out = block(x, real_edges, aug_edges, aug_types, query)
    """
    
    def __init__(
        self,
        hidden_dim: int,
        num_heads: int = 4,
        num_edge_types: int = 3,
        dropout: float = 0.1,
        use_local: bool = True,
        use_global: bool = True,
    ):
        """
        Initialize HybridBlock.
        
        Args:
            hidden_dim: Dimension of node features
            num_heads: Number of attention heads for global GT
            num_edge_types: Number of edge types (3: real, semantic, random)
            dropout: Dropout probability
            use_local: Enable local PNA branch
            use_global: Enable global GT branch
        """
        super(HybridBlock, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.use_local = use_local
        self.use_global = use_global
        
        # Local branch: Lightweight PNA on real edges
        if use_local:
            self.local_pna = LightweightPNA(
                hidden_dim=hidden_dim,
                dropout=dropout,
            )
        
        # Global branch: Sparse GT on augmented edges
        if use_global:
            self.global_gt = SparseGTConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                num_heads=num_heads,
                num_edge_types=num_edge_types,
                dropout=dropout,
            )
        
        # Fusion layer
        num_branches = int(use_local) + int(use_global)
        if num_branches > 1:
            self.fusion = nn.Sequential(
                nn.Linear(hidden_dim * num_branches, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        else:
            self.fusion = nn.Identity()
        
        # Final layer norm
        self.final_norm = nn.LayerNorm(hidden_dim)
    
    def forward(
        self,
        x: torch.Tensor,
        real_edge_index: torch.Tensor,
        aug_edge_index: torch.Tensor,
        aug_edge_type: torch.Tensor,
        query: torch.Tensor = None,
        batch: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Forward pass of hybrid block.
        
        Args:
            x: Node features [num_nodes, hidden_dim]
            real_edge_index: Real KG edges [2, num_real_edges]
            aug_edge_index: Augmented edges [2, num_aug_edges]
            aug_edge_type: Edge types [num_aug_edges]
            query: Relation query [batch_size, hidden_dim] or [num_nodes, hidden_dim]
            batch: Batch assignment for nodes (optional)
        
        Returns:
            out: Updated node features [num_nodes, hidden_dim]
        """
        outputs = []
        
        # Local branch: PNA on real edges
        if self.use_local:
            h_local = self.local_pna(x, real_edge_index, query=query, batch=batch)
            outputs.append(h_local)
        
        # Global branch: GT on augmented edges
        if self.use_global:
            h_global = self.global_gt(x, aug_edge_index, aug_edge_type)
            outputs.append(h_global)
        
        # Fusion
        if len(outputs) > 1:
            h_concat = torch.cat(outputs, dim=-1)
            h_fused = self.fusion(h_concat)
        else:
            h_fused = outputs[0]
        
        # Residual connection and final norm
        out = self.final_norm(x + h_fused)
        
        return out
    
    def __repr__(self) -> str:
        return (
            f"HybridBlock("
            f"hidden_dim={self.hidden_dim}, "
            f"heads={self.num_heads}, "
            f"local={self.use_local}, "
            f"global={self.use_global})"
        )


print("✅ LightweightPNA class defined")
print("✅ HybridBlock class defined")

✅ LightweightPNA class defined
✅ HybridBlock class defined


In [13]:
# ============================================================
# Test HybridBlock
# ============================================================

def test_hybrid_block():
    """Comprehensive test for HybridBlock module"""
    print("🧪 Testing HybridBlock...")
    print("-" * 50)
    
    # Create test data
    num_nodes = 30
    hidden_dim = 64
    batch_size = 4
    
    torch.manual_seed(42)
    
    # Node features
    x = torch.randn(num_nodes, hidden_dim)
    
    # Real edges (sparse, structural)
    real_edges = torch.tensor([
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    ], dtype=torch.long)
    
    # Augmented edges (real + semantic + random)
    aug_src = torch.randint(0, num_nodes, (150,))
    aug_dst = torch.randint(0, num_nodes, (150,))
    aug_edge_index = torch.stack([aug_src, aug_dst])
    aug_edge_type = torch.randint(0, 3, (150,))
    
    # Query (relation embedding)
    query = torch.randn(batch_size, hidden_dim)
    
    # Batch assignment
    batch = torch.repeat_interleave(torch.arange(batch_size), num_nodes // batch_size)
    batch = F.pad(batch, (0, num_nodes - len(batch)), value=batch_size - 1)
    
    # Test 1: Initialization
    print("Test 1: Initialization...")
    block = HybridBlock(
        hidden_dim=hidden_dim,
        num_heads=4,
        num_edge_types=3,
        dropout=0.1,
    )
    print(f"  Model: {block}")
    num_params = sum(p.numel() for p in block.parameters())
    print(f"  Parameters: {num_params:,}")
    print("  ✅ Initialization passed")
    
    # Test 2: Forward pass
    print("Test 2: Forward pass...")
    block.eval()
    out = block(
        x=x,
        real_edge_index=real_edges,
        aug_edge_index=aug_edge_index,
        aug_edge_type=aug_edge_type,
        query=query,
        batch=batch,
    )
    
    assert out.shape == x.shape, f"Expected shape {x.shape}, got {out.shape}"
    print(f"  Input shape: {x.shape}")
    print(f"  Output shape: {out.shape}")
    print("  ✅ Forward pass passed")
    
    # Test 3: Output validity
    print("Test 3: Output validity...")
    assert not torch.isnan(out).any(), "NaN in output!"
    assert not torch.isinf(out).any(), "Inf in output!"
    print(f"  Output range: [{out.min():.4f}, {out.max():.4f}]")
    print("  ✅ Output is valid")
    
    # Test 4: Residual connection works
    print("Test 4: Residual connection...")
    # Output should be different from input (not just passthrough)
    diff = (out - x).abs().mean()
    assert diff > 0.01, "Output too similar to input (residual not working?)"
    print(f"  Mean absolute difference: {diff:.4f}")
    print("  ✅ Residual connection working")
    
    # Test 5: Query conditioning
    print("Test 5: Query conditioning...")
    query1 = torch.randn(batch_size, hidden_dim)
    query2 = torch.randn(batch_size, hidden_dim)
    
    out1 = block(x, real_edges, aug_edge_index, aug_edge_type, query=query1, batch=batch)
    out2 = block(x, real_edges, aug_edge_index, aug_edge_type, query=query2, batch=batch)
    
    # Different queries should give different outputs
    diff = (out1 - out2).abs().mean()
    assert diff > 0.001, "Different queries give same output!"
    print(f"  Query difference effect: {diff:.4f}")
    print("  ✅ Query conditioning working")
    
    # Test 6: Gradient flow
    print("Test 6: Gradient flow...")
    block.train()
    x_grad = x.clone().requires_grad_(True)
    out = block(x_grad, real_edges, aug_edge_index, aug_edge_type, query=query, batch=batch)
    
    loss = out.sum()
    loss.backward()
    
    assert x_grad.grad is not None
    assert not torch.isnan(x_grad.grad).any()
    print(f"  Input gradient norm: {x_grad.grad.norm():.4f}")
    print("  ✅ Gradient flow working")
    
    # Test 7: Local-only mode
    print("Test 7: Local-only mode...")
    block_local = HybridBlock(hidden_dim=hidden_dim, use_local=True, use_global=False)
    out_local = block_local(x, real_edges, aug_edge_index, aug_edge_type)
    assert out_local.shape == x.shape
    print("  ✅ Local-only mode works")
    
    # Test 8: Global-only mode
    print("Test 8: Global-only mode...")
    block_global = HybridBlock(hidden_dim=hidden_dim, use_local=False, use_global=True)
    out_global = block_global(x, real_edges, aug_edge_index, aug_edge_type)
    assert out_global.shape == x.shape
    print("  ✅ Global-only mode works")
    
    # Test 9: Stacked blocks
    print("Test 9: Stacked blocks (6 layers)...")
    blocks = nn.ModuleList([
        HybridBlock(hidden_dim=hidden_dim, num_heads=4)
        for _ in range(6)
    ])
    
    h = x
    for i, blk in enumerate(blocks):
        h = blk(h, real_edges, aug_edge_index, aug_edge_type, query=query, batch=batch)
    
    assert h.shape == x.shape
    assert not torch.isnan(h).any()
    print(f"  After 6 layers - shape: {h.shape}, max: {h.abs().max():.4f}")
    print("  ✅ Stacked blocks work without explosion")
    
    # Test 10: GPU compatibility
    if torch.cuda.is_available():
        print("Test 10: GPU compatibility...")
        block_gpu = block.cuda()
        x_gpu = x.cuda()
        real_edges_gpu = real_edges.cuda()
        aug_edge_index_gpu = aug_edge_index.cuda()
        aug_edge_type_gpu = aug_edge_type.cuda()
        query_gpu = query.cuda()
        batch_gpu = batch.cuda()
        
        out_gpu = block_gpu(x_gpu, real_edges_gpu, aug_edge_index_gpu, 
                           aug_edge_type_gpu, query_gpu, batch_gpu)
        assert out_gpu.device.type == 'cuda'
        print("  ✅ GPU forward pass working")
    else:
        print("Test 10: GPU test skipped (CUDA not available)")
    
    print("-" * 50)
    print("🎉 All HybridBlock tests passed!")
    return True

# Run tests
test_hybrid_block()

🧪 Testing HybridBlock...
--------------------------------------------------
Test 1: Initialization...
  Model: HybridBlock(hidden_dim=64, heads=4, local=True, global=True)
  Parameters: 87,808
  ✅ Initialization passed
Test 2: Forward pass...
  Input shape: torch.Size([30, 64])
  Output shape: torch.Size([30, 64])
  ✅ Forward pass passed
Test 3: Output validity...
  Output range: [-3.2663, 3.6524]
  ✅ Output is valid
Test 4: Residual connection...
  Mean absolute difference: 0.4113
  ✅ Residual connection working
Test 5: Query conditioning...
  Query difference effect: 0.0421
  ✅ Query conditioning working
Test 6: Gradient flow...
  Input gradient norm: 0.0000
  ✅ Gradient flow working
Test 7: Local-only mode...
  ✅ Local-only mode works
Test 8: Global-only mode...
  ✅ Global-only mode works
Test 9: Stacked blocks (6 layers)...
  After 6 layers - shape: torch.Size([30, 64]), max: 4.1710
  ✅ Stacked blocks work without explosion
Test 10: GPU test skipped (CUDA not available)
-----------

True

# 🎯 Step 6: HybridRetriever - Main Model
The complete model integrating all components for KG reasoning

In [16]:
# ============================================================
# HybridRetriever - Main Model for KG Reasoning
# ============================================================
"""
HybridRetriever - Complete Model for Knowledge Graph Reasoning

This is the main model that integrates:
1. HybridSampler: Creates augmented graph (pre-computed)
2. PEARL_GIN: Generates positional encodings
3. HybridBlock x N: Local PNA + Global GT processing
4. Scorer: Computes final ranking scores

Architecture Overview:
---------------------

    ┌─────────────────────────────────────────────────────────┐
    │                    INPUT                                │
    │  h_index, r_index, t_index (head, relation, tail)       │
    │  hidden_states (LLM embeddings for heads)               │
    │  rel_hidden_states (LLM embeddings for relations)       │
    │  graph (with aug_edge_index, aug_edge_type)             │
    └─────────────────────────────────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                 DOWN-SCALING                            │
    │  LLM embeddings (4096) → Hidden dim (32/64)             │
    └─────────────────────────────────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                    PEARL_GIN                            │
    │  Random noise → Positional encoding                     │
    └─────────────────────────────────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │              INIT NODE FEATURES                         │
    │  x = text_embeddings                                    │
    │  x[h_index] = head_embeddings                           │
    │  x = x + positional_encoding                            │
    └─────────────────────────────────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │               HYBRID BLOCKS x N                         │
    │  for each layer:                                        │
    │    h = HybridBlock(h, real_edges, aug_edges, query)     │
    │    score = Scorer(h, query)                             │
    │    edges = select_edges(score)  # optional              │
    └─────────────────────────────────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                  FINAL SCORING                          │
    │  score = MLP(concat(h[t_index], query))                 │
    └─────────────────────────────────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                    OUTPUT                               │
    │  scores [batch_size, num_candidates]                    │
    └─────────────────────────────────────────────────────────┘

Integration with MKGL:
---------------------
This model replaces the KGLContextRetriever in the original MKGL.
The output embeddings can be projected back to LLM hidden size (4096)
and fed into the language model for generation.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.utils import degree
from typing import Tuple, Optional, Dict, Any


class Scorer(nn.Module):
    """
    Scoring module for ranking tail predictions.
    
    Computes compatibility scores between node embeddings and relation queries.
    Uses MLP with normalization for stable training.
    
    Attributes:
        hidden_dim (int): Hidden dimension
        feature_dim (int): Feature dimension after concatenation
    """
    
    def __init__(
        self,
        hidden_dim: int,
        num_mlp_layers: int = 2,
        dropout: float = 0.1,
    ):
        super(Scorer, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.feature_dim = hidden_dim * 2  # concat(hidden, query)
        
        # Linear projection
        self.linear = nn.Linear(self.feature_dim, hidden_dim)
        
        # MLP for final score
        layers = []
        for i in range(num_mlp_layers - 1):
            layers.extend([
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
        layers.append(nn.Linear(hidden_dim, 1))
        self.mlp = nn.Sequential(*layers)
        
        self._reset_parameters()
    
    def _reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(
        self,
        hidden: torch.Tensor,
        query: torch.Tensor,
        normalize: bool = True,
    ) -> torch.Tensor:
        """
        Compute scores for node embeddings given a query.
        
        Args:
            hidden: Node embeddings [num_nodes, hidden_dim] or [batch, num_candidates, hidden_dim]
            query: Relation query [batch_size, hidden_dim] or [batch, num_candidates, hidden_dim]
            normalize: Whether to normalize embeddings before scoring
        
        Returns:
            scores: Compatibility scores [num_nodes] or [batch, num_candidates]
        """
        # Normalize for stable training
        if normalize:
            hidden = F.normalize(hidden, p=2, dim=-1)
            query = F.normalize(query, p=2, dim=-1)
        
        # Handle different input shapes
        if hidden.dim() == query.dim():
            # Same dimensions - directly concatenate
            combined = torch.cat([hidden, query], dim=-1)
        elif hidden.dim() == 3 and query.dim() == 2:
            # hidden: [batch, num_candidates, dim], query: [batch, dim]
            query_expanded = query.unsqueeze(1).expand(-1, hidden.size(1), -1)
            combined = torch.cat([hidden, query_expanded], dim=-1)
        elif hidden.dim() == 2 and query.dim() == 3:
            # Unlikely but handle it
            hidden_expanded = hidden.unsqueeze(1).expand(-1, query.size(1), -1)
            combined = torch.cat([hidden_expanded, query], dim=-1)
        else:
            # Fallback: try direct concatenation
            combined = torch.cat([hidden, query], dim=-1)
        
        # Compute heuristic
        heuristic = self.linear(combined)
        heuristic = F.normalize(heuristic, p=2, dim=-1)
        
        # Final score - use hidden with same shape as heuristic
        if hidden.shape == heuristic.shape:
            score = self.mlp(hidden * heuristic).squeeze(-1)
        else:
            # If hidden was expanded, use the expanded version
            if hidden.dim() == 2 and heuristic.dim() == 3:
                hidden_for_score = hidden.unsqueeze(1).expand_as(heuristic)
            else:
                hidden_for_score = hidden
            score = self.mlp(hidden_for_score * heuristic).squeeze(-1)
        
        # Scale and clamp for stability
        score = score * 10.0
        score = torch.clamp(score, min=-15, max=15)
        
        return score


class HybridRetriever(nn.Module):
    """
    Main model for Knowledge Graph reasoning with hybrid architecture.
    
    Integrates PEARL positional encoding, HybridBlocks (PNA + GT),
    and scoring for link prediction.
    
    Attributes:
        hidden_dim (int): Hidden dimension for GNN processing
        llm_hidden_dim (int): LLM embedding dimension (e.g., 4096 for Llama-7b)
        num_layers (int): Number of HybridBlock layers
        num_relations (int): Number of relation types in KG
    
    Example:
        >>> model = HybridRetriever(
        ...     hidden_dim=32,
        ...     llm_hidden_dim=4096,
        ...     num_layers=6,
        ...     num_relations=237,
        ... )
        >>> scores = model(h_index, r_index, t_index, hidden_states, 
        ...                rel_hidden_states, graph, text_embs)
    """
    
    def __init__(
        self,
        hidden_dim: int = 32,
        llm_hidden_dim: int = 4096,
        num_layers: int = 6,
        num_relations: int = 237,
        num_heads: int = 4,
        num_edge_types: int = 3,
        dropout: float = 0.1,
        use_pearl: bool = True,
        use_edge_selection: bool = True,
        node_ratio: float = 0.1,
        degree_ratio: float = 1.0,
    ):
        """
        Initialize HybridRetriever.
        
        Args:
            hidden_dim: Hidden dimension for GNN processing (32-64 recommended)
            llm_hidden_dim: LLM embedding dimension
            num_layers: Number of HybridBlock layers (6 recommended)
            num_relations: Number of relation types in KG
            num_heads: Number of attention heads for GT
            num_edge_types: Number of edge types (3: real, semantic, random)
            dropout: Dropout probability
            use_pearl: Whether to use PEARL positional encoding
            use_edge_selection: Whether to use dynamic edge selection
            node_ratio: Ratio of nodes to select in edge selection
            degree_ratio: Ratio for degree-based edge selection
        """
        super(HybridRetriever, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.llm_hidden_dim = llm_hidden_dim
        self.num_layers = num_layers
        self.num_relations = num_relations
        self.use_pearl = use_pearl
        self.use_edge_selection = use_edge_selection
        self.node_ratio = node_ratio
        self.degree_ratio = degree_ratio
        
        # Down-scaling projections (LLM dim → GNN dim)
        self.h_down_scaling = nn.Linear(llm_hidden_dim, hidden_dim, bias=False)
        self.r_down_scaling = nn.Linear(llm_hidden_dim, hidden_dim, bias=False)
        
        # Relation embedding (learnable)
        self.rel_embedding = nn.Embedding(num_relations * 2, hidden_dim)
        
        # PEARL positional encoding
        if use_pearl:
            self.pearl = PEARL_GIN(
                input_dim=hidden_dim,
                hidden_dim=hidden_dim,
                num_layers=2,
                dropout=dropout,
            )
        
        # Hybrid blocks
        self.hybrid_layers = nn.ModuleList([
            HybridBlock(
                hidden_dim=hidden_dim,
                num_heads=num_heads,
                num_edge_types=num_edge_types,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])
        
        # Scorer
        self.scorer = Scorer(
            hidden_dim=hidden_dim,
            num_mlp_layers=2,
            dropout=dropout,
        )
        
        # Up-scaling projection (GNN dim → LLM dim) for integration with LLM
        self.up_scaling = nn.Linear(hidden_dim, llm_hidden_dim, bias=False)
        
        self._reset_parameters()
    
    def _reset_parameters(self):
        """Initialize parameters."""
        nn.init.xavier_uniform_(self.h_down_scaling.weight)
        nn.init.xavier_uniform_(self.r_down_scaling.weight)
        nn.init.xavier_uniform_(self.up_scaling.weight)
        nn.init.normal_(self.rel_embedding.weight, std=0.02)
    
    def forward(
        self,
        h_index: torch.Tensor,
        r_index: torch.Tensor,
        t_index: torch.Tensor,
        hidden_states: torch.Tensor,
        rel_hidden_states: torch.Tensor,
        graph: Data,
        text_embeddings: torch.Tensor,
        return_embeddings: bool = False,
    ) -> torch.Tensor:
        """
        Forward pass for link prediction.
        
        Args:
            h_index: Head entity indices [batch_size] or [batch_size, num_neg+1]
            r_index: Relation indices [batch_size] or [batch_size, num_neg+1]
            t_index: Tail entity indices [batch_size] or [batch_size, num_neg+1]
            hidden_states: LLM embeddings for head entities [batch_size, llm_hidden_dim]
            rel_hidden_states: LLM embeddings for relations [batch_size, llm_hidden_dim]
            graph: PyG Data object with:
                   - edge_index: Real KG edges [2, num_edges]
                   - aug_edge_index: Augmented edges [2, num_aug_edges]
                   - aug_edge_type: Edge types [num_aug_edges]
            text_embeddings: Pre-computed text embeddings [num_entities, hidden_dim]
            return_embeddings: If True, also return final node embeddings
        
        Returns:
            scores: Prediction scores [batch_size, num_candidates]
            embeddings: (optional) Node embeddings [num_nodes, hidden_dim]
        """
        device = h_index.device
        batch_size = h_index.size(0)
        num_nodes = graph.num_nodes
        
        # Handle different index shapes
        if h_index.dim() == 1:
            h_index = h_index.unsqueeze(-1)
        if r_index.dim() == 1:
            r_index = r_index.unsqueeze(-1)
        if t_index.dim() == 1:
            t_index = t_index.unsqueeze(-1)
        
        # ============================================================
        # 1. Down-scale LLM embeddings
        # ============================================================
        head_embeds = self.h_down_scaling(hidden_states.float())  # [batch, hidden_dim]
        rel_embeds = self.r_down_scaling(rel_hidden_states.float())  # [batch, hidden_dim]
        
        # Also get learnable relation embeddings
        rel_emb_learned = self.rel_embedding(r_index[:, 0])  # [batch, hidden_dim]
        
        # Combine LLM and learned relation embeddings
        query = rel_embeds + rel_emb_learned  # [batch, hidden_dim]
        
        # ============================================================
        # 2. PEARL Positional Encoding
        # ============================================================
        if self.use_pearl:
            noise = torch.randn(num_nodes, self.hidden_dim, device=device)
            h_pos = self.pearl(noise, graph.edge_index)  # [num_nodes, hidden_dim]
        else:
            h_pos = torch.zeros(num_nodes, self.hidden_dim, device=device)
        
        # ============================================================
        # 3. Initialize node features
        # ============================================================
        # Start with text embeddings
        x = text_embeddings.clone()  # [num_nodes, hidden_dim]
        
        # Inject head embeddings at head positions
        # Note: For batched processing, we need to handle this carefully
        for i in range(batch_size):
            x[h_index[i, 0]] = head_embeds[i]
        
        # Add positional encoding
        x = x + h_pos
        
        # ============================================================
        # 4. Get augmented edges from graph
        # ============================================================
        real_edge_index = graph.edge_index
        
        if hasattr(graph, 'aug_edge_index') and graph.aug_edge_index is not None:
            aug_edge_index = graph.aug_edge_index
            aug_edge_type = graph.aug_edge_type
        else:
            # Fallback: use real edges only with type 0
            aug_edge_index = real_edge_index
            aug_edge_type = torch.zeros(real_edge_index.size(1), dtype=torch.long, device=device)
        
        # ============================================================
        # 5. Hybrid message passing
        # ============================================================
        # Create batch assignment (all nodes belong to same graph for now)
        batch_assignment = torch.zeros(num_nodes, dtype=torch.long, device=device)
        
        h = x
        for layer_idx, layer in enumerate(self.hybrid_layers):
            h = layer(
                x=h,
                real_edge_index=real_edge_index,
                aug_edge_index=aug_edge_index,
                aug_edge_type=aug_edge_type,
                query=query,
                batch=batch_assignment,
            )
        
        # ============================================================
        # 6. Final scoring
        # ============================================================
        # Get tail embeddings
        # t_index: [batch_size, num_candidates]
        num_candidates = t_index.size(1)
        
        # Gather tail embeddings: [batch_size, num_candidates, hidden_dim]
        tail_embeds = h[t_index]  # PyTorch advanced indexing
        
        # Compute scores - pass query directly (2D), Scorer will expand it
        scores = self.scorer(tail_embeds, query, normalize=True)
        
        if return_embeddings:
            return scores, h
        
        return scores
    
    def get_embeddings_for_llm(
        self,
        node_embeddings: torch.Tensor,
        node_indices: torch.Tensor,
    ) -> torch.Tensor:
        """
        Project GNN embeddings back to LLM dimension.
        
        Args:
            node_embeddings: GNN node embeddings [num_nodes, hidden_dim]
            node_indices: Indices of nodes to project [batch_size]
        
        Returns:
            llm_embeddings: Embeddings in LLM space [batch_size, llm_hidden_dim]
        """
        selected = node_embeddings[node_indices]  # [batch_size, hidden_dim]
        llm_embeds = self.up_scaling(selected)  # [batch_size, llm_hidden_dim]
        return llm_embeds
    
    def __repr__(self) -> str:
        return (
            f"HybridRetriever(\n"
            f"  hidden_dim={self.hidden_dim},\n"
            f"  llm_hidden_dim={self.llm_hidden_dim},\n"
            f"  num_layers={self.num_layers},\n"
            f"  num_relations={self.num_relations},\n"
            f"  use_pearl={self.use_pearl}\n"
            f")"
        )


print("✅ Scorer class defined")
print("✅ HybridRetriever class defined")

✅ Scorer class defined
✅ HybridRetriever class defined


In [17]:
# ============================================================
# Test HybridRetriever (End-to-End)
# ============================================================

def test_hybrid_retriever():
    """Comprehensive end-to-end test for HybridRetriever"""
    print("🧪 Testing HybridRetriever (End-to-End)...")
    print("-" * 50)
    
    # Configuration
    num_entities = 100
    num_relations = 20
    hidden_dim = 32
    llm_hidden_dim = 256  # Smaller for testing (real: 4096)
    batch_size = 4
    num_candidates = 10  # 1 positive + 9 negative
    
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")
    
    # Create test graph
    num_edges = 300
    src = torch.randint(0, num_entities, (num_edges,))
    dst = torch.randint(0, num_entities, (num_edges,))
    edge_index = torch.stack([src, dst])
    
    graph = Data(
        edge_index=edge_index,
        num_nodes=num_entities,
    )
    
    # Create augmented edges (using HybridSampler)
    text_embs_full = torch.randn(num_entities, hidden_dim)
    sampler = HybridSampler(k_semantic=5, k_random=3, use_faiss=False)
    sampler.build_semantic_index(text_embs_full)
    aug_edge_index, aug_edge_type = sampler.sample_edges(graph, text_embs_full)
    
    graph.aug_edge_index = aug_edge_index
    graph.aug_edge_type = aug_edge_type
    
    # Move to device
    graph = graph.to(device)
    text_embs_full = text_embs_full.to(device)
    
    # Create batch data
    h_index = torch.randint(0, num_entities, (batch_size, 1)).to(device)
    r_index = torch.randint(0, num_relations, (batch_size, 1)).to(device)
    t_index = torch.randint(0, num_entities, (batch_size, num_candidates)).to(device)
    
    # Simulated LLM embeddings
    hidden_states = torch.randn(batch_size, llm_hidden_dim).to(device)
    rel_hidden_states = torch.randn(batch_size, llm_hidden_dim).to(device)
    
    # Test 1: Initialization
    print("Test 1: Initialization...")
    model = HybridRetriever(
        hidden_dim=hidden_dim,
        llm_hidden_dim=llm_hidden_dim,
        num_layers=6,
        num_relations=num_relations,
        num_heads=4,
        dropout=0.1,
    ).to(device)
    
    print(f"  Model:\n{model}")
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {num_params:,}")
    print("  ✅ Initialization passed")
    
    # Test 2: Forward pass
    print("Test 2: Forward pass...")
    model.eval()
    with torch.no_grad():
        scores = model(
            h_index=h_index,
            r_index=r_index,
            t_index=t_index,
            hidden_states=hidden_states,
            rel_hidden_states=rel_hidden_states,
            graph=graph,
            text_embeddings=text_embs_full,
        )
    
    expected_shape = (batch_size, num_candidates)
    assert scores.shape == expected_shape, f"Expected {expected_shape}, got {scores.shape}"
    print(f"  Input: batch_size={batch_size}, candidates={num_candidates}")
    print(f"  Output scores shape: {scores.shape}")
    print(f"  Scores range: [{scores.min():.4f}, {scores.max():.4f}]")
    print("  ✅ Forward pass passed")
    
    # Test 3: Output validity
    print("Test 3: Output validity...")
    assert not torch.isnan(scores).any(), "NaN in scores!"
    assert not torch.isinf(scores).any(), "Inf in scores!"
    assert scores.abs().max() <= 15, f"Scores exceed clamp range: {scores.abs().max()}"
    print("  ✅ Scores are valid and clamped")
    
    # Test 4: Gradient flow
    print("Test 4: Gradient flow...")
    model.train()
    hidden_states_grad = hidden_states.clone().requires_grad_(True)
    
    scores = model(
        h_index=h_index,
        r_index=r_index,
        t_index=t_index,
        hidden_states=hidden_states_grad,
        rel_hidden_states=rel_hidden_states,
        graph=graph,
        text_embeddings=text_embs_full,
    )
    
    # Compute simple loss
    labels = torch.zeros(batch_size, dtype=torch.long, device=device)  # First candidate is positive
    loss = F.cross_entropy(scores, labels)
    loss.backward()
    
    assert hidden_states_grad.grad is not None
    assert not torch.isnan(hidden_states_grad.grad).any()
    print(f"  Loss: {loss.item():.4f}")
    print(f"  Input gradient norm: {hidden_states_grad.grad.norm():.4f}")
    print("  ✅ Gradient flow working")
    
    # Test 5: Return embeddings
    print("Test 5: Return embeddings...")
    model.eval()
    with torch.no_grad():
        scores, embeddings = model(
            h_index=h_index,
            r_index=r_index,
            t_index=t_index,
            hidden_states=hidden_states,
            rel_hidden_states=rel_hidden_states,
            graph=graph,
            text_embeddings=text_embs_full,
            return_embeddings=True,
        )
    
    assert embeddings.shape == (num_entities, hidden_dim)
    print(f"  Embeddings shape: {embeddings.shape}")
    print("  ✅ Embeddings returned correctly")
    
    # Test 6: LLM integration
    print("Test 6: LLM integration (up-scaling)...")
    node_indices = torch.randint(0, num_entities, (batch_size,), device=device)
    llm_embeds = model.get_embeddings_for_llm(embeddings, node_indices)
    
    assert llm_embeds.shape == (batch_size, llm_hidden_dim)
    print(f"  Projected embeddings shape: {llm_embeds.shape}")
    print("  ✅ LLM integration working")
    
    # Test 7: Different batch sizes
    print("Test 7: Different batch sizes...")
    for bs in [1, 8, 16]:
        h_idx = torch.randint(0, num_entities, (bs, 1), device=device)
        r_idx = torch.randint(0, num_relations, (bs, 1), device=device)
        t_idx = torch.randint(0, num_entities, (bs, num_candidates), device=device)
        hs = torch.randn(bs, llm_hidden_dim, device=device)
        rhs = torch.randn(bs, llm_hidden_dim, device=device)
        
        with torch.no_grad():
            sc = model(h_idx, r_idx, t_idx, hs, rhs, graph, text_embs_full)
        assert sc.shape == (bs, num_candidates)
        print(f"  Batch size {bs}: ✓")
    print("  ✅ Different batch sizes work")
    
    # Test 8: Without PEARL
    print("Test 8: Model without PEARL...")
    model_no_pearl = HybridRetriever(
        hidden_dim=hidden_dim,
        llm_hidden_dim=llm_hidden_dim,
        num_layers=3,
        num_relations=num_relations,
        use_pearl=False,
    ).to(device)
    
    with torch.no_grad():
        scores_no_pearl = model_no_pearl(
            h_index, r_index, t_index,
            hidden_states, rel_hidden_states,
            graph, text_embs_full,
        )
    assert scores_no_pearl.shape == expected_shape
    print("  ✅ Model without PEARL works")
    
    # Test 9: Graph without augmentation
    print("Test 9: Graph without augmentation...")
    graph_no_aug = Data(
        edge_index=edge_index.to(device),
        num_nodes=num_entities,
    )
    
    with torch.no_grad():
        scores_no_aug = model(
            h_index, r_index, t_index,
            hidden_states, rel_hidden_states,
            graph_no_aug, text_embs_full,
        )
    assert scores_no_aug.shape == expected_shape
    print("  ✅ Fallback without augmented edges works")
    
    # Test 10: Memory check
    print("Test 10: Memory efficiency...")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        
        # Run a few forward passes
        for _ in range(5):
            with torch.no_grad():
                _ = model(h_index, r_index, t_index, hidden_states, 
                         rel_hidden_states, graph, text_embs_full)
        
        peak_memory = torch.cuda.max_memory_allocated() / 1024**2
        print(f"  Peak GPU memory: {peak_memory:.2f} MB")
    else:
        print("  Memory test skipped (CUDA not available)")
    print("  ✅ Memory efficiency check done")
    
    print("-" * 50)
    print("🎉 All HybridRetriever tests passed!")
    print(f"\n📊 Summary:")
    print(f"  - Total parameters: {num_params:,}")
    print(f"  - Hidden dim: {hidden_dim}")
    print(f"  - LLM dim: {llm_hidden_dim}")
    print(f"  - Layers: 6")
    print(f"  - Device: {device}")
    return True

# Run tests
test_hybrid_retriever()

🧪 Testing HybridRetriever (End-to-End)...
--------------------------------------------------
  Device: cpu
Test 1: Initialization...
  Model:
HybridRetriever(
  hidden_dim=32,
  llm_hidden_dim=256,
  num_layers=6,
  num_relations=20,
  use_pearl=True
)
  Total parameters: 167,907
  ✅ Initialization passed
Test 2: Forward pass...
  Input: batch_size=4, candidates=10
  Output scores shape: torch.Size([4, 10])
  Scores range: [-0.2686, 0.6139]
  ✅ Forward pass passed
Test 3: Output validity...
  ✅ Scores are valid and clamped
Test 4: Gradient flow...
  Loss: 2.4089
  Input gradient norm: 0.0510
  ✅ Gradient flow working
Test 5: Return embeddings...
  Embeddings shape: torch.Size([100, 32])
  ✅ Embeddings returned correctly
Test 6: LLM integration (up-scaling)...
  Projected embeddings shape: torch.Size([4, 256])
  ✅ LLM integration working
Test 7: Different batch sizes...
  Batch size 1: ✓
  Batch size 8: ✓
  Batch size 16: ✓
  ✅ Different batch sizes work
Test 8: Model without PEARL...
 

True

# 📈 Step 7: Training Pipeline
Loss functions, metrics, and training loop

In [18]:
# ============================================================
# Training Pipeline: Loss Functions and Metrics
# ============================================================
"""
Training utilities for HybridRetriever

Loss Functions:
--------------
1. ContrastiveLoss: Margin-based ranking loss
   - Encourages positive score > negative scores by margin
   
2. CrossEntropyLoss: Standard classification loss
   - Treats as multi-class classification (positive vs negatives)
   
3. HybridLoss: Combined loss with regularization
   - Combines ranking loss + regularization terms

Metrics:
--------
- Hits@K: Percentage of correct predictions in top K
- MRR: Mean Reciprocal Rank
- MR: Mean Rank
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Tuple, Optional


class ContrastiveLoss(nn.Module):
    """
    Margin-based contrastive loss for ranking.
    
    Loss = max(0, margin - positive_score + negative_score)
    
    This encourages the model to rank positive samples higher
    than negative samples by at least the margin.
    """
    
    def __init__(self, margin: float = 1.0, reduction: str = 'mean'):
        super().__init__()
        self.margin = margin
        self.reduction = reduction
    
    def forward(
        self,
        pos_scores: torch.Tensor,
        neg_scores: torch.Tensor,
    ) -> torch.Tensor:
        """
        Compute contrastive loss.
        
        Args:
            pos_scores: Positive sample scores [batch_size, 1] or [batch_size]
            neg_scores: Negative sample scores [batch_size, num_neg]
        
        Returns:
            loss: Scalar loss value
        """
        # Ensure pos_scores is [batch_size, 1]
        if pos_scores.dim() == 1:
            pos_scores = pos_scores.unsqueeze(-1)
        
        # Compute margin ranking loss
        # margin - pos + neg should be < 0 for correct ranking
        loss = F.relu(self.margin - pos_scores + neg_scores)
        
        # Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


class HybridLoss(nn.Module):
    """
    Combined loss for HybridRetriever training.
    
    Combines:
    1. Primary loss (contrastive or cross-entropy)
    2. Regularization on relation embeddings
    3. Diversity regularization on edge type embeddings
    """
    
    def __init__(
        self,
        loss_type: str = 'cross_entropy',  # 'contrastive' or 'cross_entropy'
        margin: float = 1.0,
        alpha_reg: float = 0.01,
        alpha_diversity: float = 0.001,
    ):
        """
        Initialize HybridLoss.
        
        Args:
            loss_type: Type of primary loss ('contrastive' or 'cross_entropy')
            margin: Margin for contrastive loss
            alpha_reg: Weight for L2 regularization on embeddings
            alpha_diversity: Weight for diversity regularization
        """
        super().__init__()
        self.loss_type = loss_type
        self.margin = margin
        self.alpha_reg = alpha_reg
        self.alpha_diversity = alpha_diversity
        
        if loss_type == 'contrastive':
            self.criterion = ContrastiveLoss(margin=margin)
        else:
            self.criterion = nn.CrossEntropyLoss()
    
    def forward(
        self,
        scores: torch.Tensor,
        labels: torch.Tensor,
        model: nn.Module = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute total loss with regularization.
        
        Args:
            scores: Prediction scores [batch_size, num_candidates]
            labels: Ground truth labels [batch_size] (index of positive)
                   or [batch_size, num_candidates] (binary mask)
            model: HybridRetriever model (for regularization)
        
        Returns:
            total_loss: Combined loss
            loss_dict: Dictionary with individual loss components
        """
        loss_dict = {}
        
        # Primary loss
        if self.loss_type == 'contrastive':
            # Separate positive and negative scores
            batch_size = scores.size(0)
            pos_scores = scores[torch.arange(batch_size), labels].unsqueeze(-1)
            
            # Create negative mask
            neg_mask = torch.ones_like(scores, dtype=torch.bool)
            neg_mask[torch.arange(batch_size), labels] = False
            neg_scores = scores[neg_mask].view(batch_size, -1)
            
            loss_primary = self.criterion(pos_scores, neg_scores)
        else:
            loss_primary = self.criterion(scores, labels)
        
        loss_dict['loss_primary'] = loss_primary.item()
        total_loss = loss_primary
        
        # Regularization
        if model is not None and self.alpha_reg > 0:
            # L2 regularization on relation embeddings
            if hasattr(model, 'rel_embedding'):
                rel_emb = model.rel_embedding.weight
                loss_reg = torch.norm(rel_emb, p=2) * self.alpha_reg
                loss_dict['loss_reg'] = loss_reg.item()
                total_loss = total_loss + loss_reg
            
            # Diversity regularization on edge type embeddings
            if self.alpha_diversity > 0:
                loss_diversity = torch.tensor(0.0, device=scores.device)
                for layer in model.hybrid_layers:
                    if hasattr(layer, 'global_gt') and hasattr(layer.global_gt, 'edge_type_emb'):
                        emb = layer.global_gt.edge_type_emb.weight
                        # Penalize similar embeddings
                        sim = torch.mm(emb, emb.t())
                        loss_diversity += torch.triu(sim, diagonal=1).abs().mean()
                
                loss_diversity = loss_diversity * self.alpha_diversity
                loss_dict['loss_diversity'] = loss_diversity.item()
                total_loss = total_loss + loss_diversity
        
        loss_dict['loss_total'] = total_loss.item()
        
        return total_loss, loss_dict


def compute_ranking_metrics(
    scores: torch.Tensor,
    labels: torch.Tensor,
    ks: list = [1, 3, 10],
) -> Dict[str, float]:
    """
    Compute ranking metrics for evaluation.
    
    Args:
        scores: Prediction scores [batch_size, num_candidates]
        labels: Ground truth indices [batch_size]
        ks: List of K values for Hits@K
    
    Returns:
        metrics: Dictionary with Hits@K, MRR, MR
    """
    batch_size = scores.size(0)
    
    # Sort scores in descending order and get ranks
    sorted_indices = torch.argsort(scores, dim=-1, descending=True)
    
    # Find rank of positive (1-indexed)
    ranks = torch.zeros(batch_size, device=scores.device)
    for i in range(batch_size):
        rank = (sorted_indices[i] == labels[i]).nonzero(as_tuple=True)[0]
        ranks[i] = rank[0].float() + 1 if len(rank) > 0 else scores.size(1)
    
    metrics = {}
    
    # Hits@K
    for k in ks:
        hits_k = (ranks <= k).float().mean()
        metrics[f'hits@{k}'] = hits_k.item()
    
    # MRR (Mean Reciprocal Rank)
    mrr = (1.0 / ranks).mean()
    metrics['mrr'] = mrr.item()
    
    # MR (Mean Rank)
    mr = ranks.mean()
    metrics['mr'] = mr.item()
    
    return metrics


print("✅ ContrastiveLoss class defined")
print("✅ HybridLoss class defined")
print("✅ compute_ranking_metrics function defined")

✅ ContrastiveLoss class defined
✅ HybridLoss class defined
✅ compute_ranking_metrics function defined


In [19]:
# ============================================================
# Test Training Pipeline
# ============================================================

def test_training_pipeline():
    """Test loss functions and metrics"""
    print("🧪 Testing Training Pipeline...")
    print("-" * 50)
    
    torch.manual_seed(42)
    batch_size = 8
    num_candidates = 50
    
    # Simulated scores and labels
    scores = torch.randn(batch_size, num_candidates)
    labels = torch.zeros(batch_size, dtype=torch.long)  # First position is positive
    
    # Make positive scores higher (for realistic test)
    scores[:, 0] += 2.0  # Boost positive scores
    
    # Test 1: ContrastiveLoss
    print("Test 1: ContrastiveLoss...")
    contrastive_loss = ContrastiveLoss(margin=1.0)
    pos_scores = scores[:, 0:1]
    neg_scores = scores[:, 1:]
    loss_c = contrastive_loss(pos_scores, neg_scores)
    print(f"  Contrastive loss: {loss_c.item():.4f}")
    assert not torch.isnan(loss_c)
    print("  ✅ ContrastiveLoss working")
    
    # Test 2: CrossEntropyLoss
    print("Test 2: CrossEntropyLoss...")
    ce_loss = F.cross_entropy(scores, labels)
    print(f"  CrossEntropy loss: {ce_loss.item():.4f}")
    assert not torch.isnan(ce_loss)
    print("  ✅ CrossEntropyLoss working")
    
    # Test 3: HybridLoss
    print("Test 3: HybridLoss...")
    hybrid_loss = HybridLoss(loss_type='cross_entropy', alpha_reg=0.01)
    total_loss, loss_dict = hybrid_loss(scores, labels, model=None)
    print(f"  Total loss: {total_loss.item():.4f}")
    print(f"  Loss components: {loss_dict}")
    assert not torch.isnan(total_loss)
    print("  ✅ HybridLoss working")
    
    # Test 4: Ranking metrics
    print("Test 4: Ranking metrics...")
    metrics = compute_ranking_metrics(scores, labels, ks=[1, 3, 10])
    print(f"  Metrics: {metrics}")
    
    assert 0 <= metrics['hits@1'] <= 1
    assert 0 <= metrics['mrr'] <= 1
    assert metrics['mr'] >= 1
    print("  ✅ Ranking metrics working")
    
    # Test 5: Perfect predictions
    print("Test 5: Perfect predictions...")
    perfect_scores = torch.zeros(batch_size, num_candidates)
    perfect_scores[:, 0] = 10.0  # Positive is highest
    
    perfect_metrics = compute_ranking_metrics(perfect_scores, labels)
    assert perfect_metrics['hits@1'] == 1.0
    assert perfect_metrics['mrr'] == 1.0
    assert perfect_metrics['mr'] == 1.0
    print(f"  Perfect Hits@1: {perfect_metrics['hits@1']:.4f}")
    print("  ✅ Perfect prediction metrics correct")
    
    # Test 6: Gradient flow with HybridLoss
    print("Test 6: Gradient flow with loss...")
    scores_grad = scores.clone().requires_grad_(True)
    total_loss, _ = hybrid_loss(scores_grad, labels)
    total_loss.backward()
    
    assert scores_grad.grad is not None
    assert not torch.isnan(scores_grad.grad).any()
    print(f"  Gradient norm: {scores_grad.grad.norm():.4f}")
    print("  ✅ Gradient flow working")
    
    print("-" * 50)
    print("🎉 All training pipeline tests passed!")
    return True

# Run tests
test_training_pipeline()

🧪 Testing Training Pipeline...
--------------------------------------------------
Test 1: ContrastiveLoss...
  Contrastive loss: 0.0816
  ✅ ContrastiveLoss working
Test 2: CrossEntropyLoss...
  CrossEntropy loss: 2.1961
  ✅ CrossEntropyLoss working
Test 3: HybridLoss...
  Total loss: 2.1961
  Loss components: {'loss_primary': 2.1961019039154053, 'loss_total': 2.1961019039154053}
  ✅ HybridLoss working
Test 4: Ranking metrics...
  Metrics: {'hits@1': 0.625, 'hits@3': 0.75, 'hits@10': 1.0, 'mrr': 0.7209821343421936, 'mr': 2.75}
  ✅ Ranking metrics working
Test 5: Perfect predictions...
  Perfect Hits@1: 1.0000
  ✅ Perfect prediction metrics correct
Test 6: Gradient flow with loss...
  Gradient norm: 0.3128
  ✅ Gradient flow working
--------------------------------------------------
🎉 All training pipeline tests passed!


True

# ✅ Step 8: Run All Tests
Execute this cell to run all module tests sequentially

In [20]:
# ============================================================
# Run All Tests
# ============================================================

def run_all_tests():
    """Run all module tests and report results"""
    print("=" * 60)
    print("🚀 HYBRID GNN - COMPLETE TEST SUITE")
    print("=" * 60)
    
    results = {}
    
    # Test 1: HybridSampler
    print("\n" + "=" * 60)
    try:
        results['HybridSampler'] = test_hybrid_sampler()
    except Exception as e:
        print(f"❌ HybridSampler FAILED: {e}")
        results['HybridSampler'] = False
    
    # Test 2: PEARL_GIN
    print("\n" + "=" * 60)
    try:
        results['PEARL_GIN'] = test_pearl_gin()
    except Exception as e:
        print(f"❌ PEARL_GIN FAILED: {e}")
        results['PEARL_GIN'] = False
    
    # Test 3: SparseGTConv
    print("\n" + "=" * 60)
    try:
        results['SparseGTConv'] = test_sparse_gt_conv()
    except Exception as e:
        print(f"❌ SparseGTConv FAILED: {e}")
        results['SparseGTConv'] = False
    
    # Test 4: HybridBlock
    print("\n" + "=" * 60)
    try:
        results['HybridBlock'] = test_hybrid_block()
    except Exception as e:
        print(f"❌ HybridBlock FAILED: {e}")
        results['HybridBlock'] = False
    
    # Test 5: HybridRetriever
    print("\n" + "=" * 60)
    try:
        results['HybridRetriever'] = test_hybrid_retriever()
    except Exception as e:
        print(f"❌ HybridRetriever FAILED: {e}")
        results['HybridRetriever'] = False
    
    # Test 6: Training Pipeline
    print("\n" + "=" * 60)
    try:
        results['TrainingPipeline'] = test_training_pipeline()
    except Exception as e:
        print(f"❌ TrainingPipeline FAILED: {e}")
        results['TrainingPipeline'] = False
    
    # Summary
    print("\n" + "=" * 60)
    print("📊 TEST SUMMARY")
    print("=" * 60)
    
    passed = sum(results.values())
    total = len(results)
    
    for module, status in results.items():
        icon = "✅" if status else "❌"
        print(f"  {icon} {module}")
    
    print("-" * 60)
    print(f"  Total: {passed}/{total} passed")
    
    if passed == total:
        print("\n🎉 ALL TESTS PASSED! Ready for training.")
    else:
        print(f"\n⚠️ {total - passed} test(s) failed. Please fix before training.")
    
    return results

# Run all tests
test_results = run_all_tests()

🚀 HYBRID GNN - COMPLETE TEST SUITE

🧪 Testing HybridSampler...
--------------------------------------------------
Test 1: Initialization...
  ✅ Initialization passed
Test 2: Build semantic index...
  ✅ Semantic index built
Test 3: Sample edges...
  Total edges: 72
  ✅ Edge sampling passed
Test 4: Edge type validity...
  ✅ Edge types are valid (0, 1, or 2)
Test 5: Real edges preservation...
  Original edges: 22, Preserved: 22
  ✅ Real edges preserved
Test 6: No self-loops in augmented edges...
  ✅ No self-loops found
Test 7: Valid node indices...
  ✅ All node indices are valid
Test 8: Edge statistics...
  Total: 72
  Real: 22 (30.6%)
  Semantic: 30 (41.7%)
  Random: 20 (27.8%)
  ✅ Statistics computed
Test 9: Semantic edges connect similar nodes...
  Intra-cluster edges (nodes 0,1,2): 6
  ✅ Semantic clustering working
--------------------------------------------------
🎉 All HybridSampler tests passed!

🧪 Testing PEARL_GIN...
--------------------------------------------------
Test 1: Init

## Run All Tests
Execute this cell to run all module tests sequentially.

In [21]:
# ============================================================
# RUN ALL TESTS
# ============================================================

def run_all_tests():
    """Execute all module tests in sequence."""
    print("=" * 70)
    print("HYBRID GNN MODULE TESTS")
    print("=" * 70)
    
    tests = [
        ("HybridSampler", test_hybrid_sampler),
        ("PEARL_GIN", test_pearl_gin),
        ("SparseGTConv", test_sparse_gt_conv),
        ("HybridBlock", test_hybrid_block),
        ("HybridRetriever", test_hybrid_retriever),
        ("Training Pipeline", test_training_pipeline),
    ]
    
    results = {}
    
    for name, test_fn in tests:
        print(f"\n{'=' * 70}")
        print(f"Testing: {name}")
        print("=" * 70)
        
        try:
            test_fn()
            results[name] = "✅ PASSED"
            print(f"\n{name}: ✅ PASSED")
        except Exception as e:
            results[name] = f"❌ FAILED: {str(e)}"
            print(f"\n{name}: ❌ FAILED")
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
    
    # Summary
    print("\n" + "=" * 70)
    print("TEST SUMMARY")
    print("=" * 70)
    
    passed = sum(1 for v in results.values() if "PASSED" in v)
    failed = len(results) - passed
    
    for name, result in results.items():
        print(f"  {name}: {result}")
    
    print(f"\nTotal: {passed}/{len(results)} passed")
    
    if failed == 0:
        print("\n🎉 All tests passed! Ready to proceed with training integration.")
    else:
        print(f"\n⚠️ {failed} test(s) failed. Please fix before proceeding.")
    
    return results

# Run all tests
test_results = run_all_tests()

HYBRID GNN MODULE TESTS

Testing: HybridSampler
🧪 Testing HybridSampler...
--------------------------------------------------
Test 1: Initialization...
  ✅ Initialization passed
Test 2: Build semantic index...
  ✅ Semantic index built
Test 3: Sample edges...
  Total edges: 72
  ✅ Edge sampling passed
Test 4: Edge type validity...
  ✅ Edge types are valid (0, 1, or 2)
Test 5: Real edges preservation...
  Original edges: 22, Preserved: 22
  ✅ Real edges preserved
Test 6: No self-loops in augmented edges...
  ✅ No self-loops found
Test 7: Valid node indices...
  ✅ All node indices are valid
Test 8: Edge statistics...
  Total: 72
  Real: 22 (30.6%)
  Semantic: 30 (41.7%)
  Random: 20 (27.8%)
  ✅ Statistics computed
Test 9: Semantic edges connect similar nodes...
  Intra-cluster edges (nodes 0,1,2): 6
  ✅ Semantic clustering working
--------------------------------------------------
🎉 All HybridSampler tests passed!

HybridSampler: ✅ PASSED

Testing: PEARL_GIN
🧪 Testing PEARL_GIN...
-------

In [22]:
!python preprocess_new.py -c config/fb15k237_ind_hybrid.yaml -v v1

python3: can't open file '/content/preprocess_new.py': [Errno 2] No such file or directory
